# Phase 1 Foundation — Interactive Validation

This notebook exercises every piece of Phase 1 without calling any LLM or external service.
Run each section in order. Every cell ends with assertions so failures surface immediately.

**Sections**
1. Setup & imports
2. Timestamp utility (`utcnow_iso`)
3. WorkflowState types
4. Agent output schemas (all 8)
5. Database initialisation — 18 tables
6. WorkflowRepository
7. JobRepository
8. ScoreRepository
9. StepRepository
10. ObservabilityRepository
11. ConfigService
12. Retention purge
13. End-to-end mini-workflow simulation
14. Cleanup

---
## 1. Setup & Imports

In [6]:
import os
import re
import sys
import time
import uuid
from pathlib import Path
from pprint import pprint

# --- Project root resolution ---
# Works whether Jupyter is launched from the project root or from notebooks/
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# --- Isolated test database (never touches data/v2.db) ---
TEMP_DB = Path("data/notebook_test.db")
if TEMP_DB.exists():
    TEMP_DB.unlink()

print(f"Project root : {project_root}")
print(f"Test DB path : {TEMP_DB}")
print("Ready.")

Project root : c:\dev\github\jobsearchagent-v2
Test DB path : data\notebook_test.db
Ready.


---
## 2. Timestamp Utility

`utcnow_iso()` is the single source of truth for all timestamps in the system.
Every repository uses it. Nothing else may produce timestamps.

In [7]:
from app.repositories.database import utcnow_iso

ts1 = utcnow_iso()
time.sleep(0.05)
ts2 = utcnow_iso()
time.sleep(0.05)
ts3 = utcnow_iso()

print("Three consecutive timestamps:")
for ts in [ts1, ts2, ts3]:
    print(f"  {ts}")

# Format: YYYY-MM-DDTHH:MM:SS.mmmZ  (millisecond precision, always UTC)
pattern = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}Z$"
for ts in [ts1, ts2, ts3]:
    assert re.match(pattern, ts), f"Bad format: {ts}"

# Lexicographic sort order must equal chronological order
assert ts1 < ts2 < ts3, "Timestamps do not sort chronologically"

print()
print("✓ Format is ISO 8601 UTC with millisecond precision")
print("✓ Lexicographic sort = chronological sort")

Three consecutive timestamps:
  2026-04-29T17:06:04.645Z
  2026-04-29T17:06:04.695Z
  2026-04-29T17:06:04.746Z

✓ Format is ISO 8601 UTC with millisecond precision
✓ Lexicographic sort = chronological sort


---
## 3. WorkflowState Types

Six types live in `app/state/workflow_state.py`. Each is validated here.

In [8]:
from app.state.workflow_state import (
    WorkflowStatus, WorkflowStep,
    StepExecution, RunMetrics, WorkflowError, HumanDecision,
    WorkflowState,
)

# --- WorkflowStatus ---
print("WorkflowStatus values:")
for s in WorkflowStatus:
    print(f"  {s.value}")

# --- WorkflowStep ---
steps = list(WorkflowStep)
print(f"\nWorkflowStep: {len(steps)} values")
for s in steps:
    print(f"  {s.value}")

WorkflowStatus values:
  initialized
  running
  waiting_for_user
  completed
  failed
  cancelled

WorkflowStep: 17 values
  initialized
  job_discovery
  resume_profile_loading
  scoring
  awaiting_job_selection
  research
  resume_critique
  review_audit
  reflection_decision
  career_advice
  interview_prep
  tailoring
  fidelity_review
  awaiting_user_approval
  report_generation
  completed
  failed


In [10]:
# --- StepExecution ---
now = utcnow_iso()
step_ex = StepExecution(
    step=WorkflowStep.SCORING,
    status="started",
    started_at=now,
)
assert step_ex.completed_at is None
assert step_ex.duration_ms is None
print(f"StepExecution (started): step={step_ex.step.value}, status={step_ex.status}")

step_ex_done = StepExecution(
    step=WorkflowStep.SCORING,
    status="completed",
    started_at=now,
    completed_at=utcnow_iso(),
    duration_ms=214,
    notes="Batch scored 5 jobs",
)
assert step_ex_done.duration_ms == 214
print(f"StepExecution (done):    duration_ms={step_ex_done.duration_ms}, notes='{step_ex_done.notes}'")

# --- RunMetrics ---
metrics = RunMetrics()
assert metrics.llm_calls == 0
assert metrics.estimated_cost_usd == 0.0
assert metrics.started_at is None
print(f"\nRunMetrics defaults: {metrics.model_dump()}")

# --- WorkflowError ---
err = WorkflowError(
    step="scoring",
    error_type="ScrapeError",
    message="LinkedIn returned 429 Too Many Requests",
    recoverable=True,
    occurred_at=utcnow_iso(),
    suggested_action="Wait 60s and retry",
)
assert err.recoverable is True
print(f"\nWorkflowError: type={err.error_type}, recoverable={err.recoverable}")

# --- HumanDecision ---
presented = utcnow_iso()
time.sleep(0.1)
decided = utcnow_iso()
decision = HumanDecision(
    decision_type="select_jobs_for_deep_review",
    decision_value="confirmed",
    payload={"selected_job_ids": ["job-001", "job-002"]},
    presented_at=presented,
    decided_at=decided,
)
assert decision.decided_at > decision.presented_at
print(f"\nHumanDecision: type={decision.decision_type}")
print(f"  presented_at: {decision.presented_at}")
print(f"  decided_at:   {decision.decided_at}")
print("✓ decided_at > presented_at")

print("\n✓ StepExecution, RunMetrics, WorkflowError, HumanDecision all valid")

StepExecution (started): step=scoring, status=started
StepExecution (done):    duration_ms=214, notes='Batch scored 5 jobs'

RunMetrics defaults: {'llm_calls': 0, 'tokens_input': 0, 'tokens_output': 0, 'estimated_cost_usd': 0.0, 'total_duration_ms': 0, 'started_at': None, 'completed_at': None}

WorkflowError: type=ScrapeError, recoverable=True

HumanDecision: type=select_jobs_for_deep_review
  presented_at: 2026-04-29T17:06:37.165Z
  decided_at:   2026-04-29T17:06:37.266Z
✓ decided_at > presented_at

✓ StepExecution, RunMetrics, WorkflowError, HumanDecision all valid


In [11]:
# --- WorkflowState (the main container) ---
now = utcnow_iso()
state = WorkflowState(
    workflow_id="wf-notebook-001",
    workflow_type="job_search",
    status=WorkflowStatus.INITIALIZED,
    current_step=WorkflowStep.INITIALIZED,
    search_criteria={"roles": ["Staff Engineer"], "locations": ["Remote"]},
    created_at=now,
    updated_at=now,
)

print(f"WorkflowState: id={state.workflow_id}, status={state.status.value}")
print(f"  raw_jobs:   {state.raw_jobs} (empty list by default)")
print(f"  errors:     {state.errors} (empty list by default)")
print(f"  run_metrics: {state.run_metrics.model_dump()}")

# Append a step and error (state is mutable before it enters the DB)
state.step_history.append(step_ex)
state.errors.append(err)
state.human_decisions.append(decision)
print(f"  step_history: {len(state.step_history)} entry")
print(f"  errors:       {len(state.errors)} entry")
print(f"  human_decisions: {len(state.human_decisions)} entry")

# JSON round-trip
state_dict = state.model_dump()
state_restored = WorkflowState.model_validate(state_dict)
assert state_restored.workflow_id == state.workflow_id
assert state_restored.status == state.status
assert len(state_restored.step_history) == 1
assert state_restored.step_history[0].step == WorkflowStep.SCORING
assert len(state_restored.human_decisions) == 1
assert state_restored.human_decisions[0].decided_at == decision.decided_at

print("\n✓ WorkflowState JSON round-trip: model_dump → model_validate preserves all fields")

WorkflowState: id=wf-notebook-001, status=initialized
  raw_jobs:   [] (empty list by default)
  errors:     [] (empty list by default)
  run_metrics: {'llm_calls': 0, 'tokens_input': 0, 'tokens_output': 0, 'estimated_cost_usd': 0.0, 'total_duration_ms': 0, 'started_at': None, 'completed_at': None}
  step_history: 1 entry
  errors:       1 entry
  human_decisions: 1 entry

✓ WorkflowState JSON round-trip: model_dump → model_validate preserves all fields


---
## 4. Agent Output Schemas

All 8 schemas from `app/schemas/`. Each is constructed with valid data and tested for
rejection of invalid data (out-of-range scores, missing required fields).

In [12]:
from pydantic import ValidationError

from app.schemas.job_score import JobScore
from app.schemas.research_context import ResearchContext, ResearchStep
from app.schemas.resume_review import ResumeReview, SectionReview
from app.schemas.review_audit import ReviewAudit
from app.schemas.career_advice import CareerAdvice
from app.schemas.interview_prep import InterviewPrep
from app.schemas.tailored_resume_draft import TailoredResumeDraft, TailoredBullet
from app.schemas.fidelity_review import FidelityReview

print("All 8 agent output schemas imported successfully.")

All 8 agent output schemas imported successfully.


In [13]:
# --- 1. JobScore ---
score = JobScore(
    job_id="job-001",
    resume_id="res-001",
    overall_score=82,
    technical_score=85,
    architecture_score=80,
    leadership_score=78,
    domain_score=75,
    match_summary="Strong technical and architecture fit for a Staff Engineer role.",
    strengths=["GCP expertise", "Kubernetes at scale", "Python backend systems"],
    gaps=["No direct fintech domain experience", "Limited public cloud cost optimisation work"],
    recommended_next_action="shortlist_for_deep_review",
    confidence=88,
)
print(f"JobScore: overall={score.overall_score}, technical={score.technical_score}, confidence={score.confidence}")

# Reject score > 100
try:
    JobScore(**{**score.model_dump(), "overall_score": 150})
    assert False, "Should have raised ValidationError"
except ValidationError as e:
    print(f"✓ Rejects overall_score=150 → {e.errors()[0]['msg']}")

# Reject score < 0
try:
    JobScore(**{**score.model_dump(), "confidence": -1})
    assert False
except ValidationError:
    print("✓ Rejects confidence=-1")

# Reject missing required field
try:
    data = score.model_dump()
    del data["match_summary"]
    JobScore(**data)
    assert False
except ValidationError:
    print("✓ Rejects missing match_summary")

JobScore: overall=82, technical=85, confidence=88
✓ Rejects overall_score=150 → Input should be less than or equal to 100
✓ Rejects confidence=-1
✓ Rejects missing match_summary


In [14]:
# --- 2. ResearchContext ---
research = ResearchContext(
    job_id="job-001",
    company_summary="TechCorp is a Series C fintech company with 800 engineers.",
    role_context="Senior IC role in the platform team, owns the data pipeline infrastructure.",
    technology_signals=["Kubernetes", "GCP", "Kafka", "Python"],
    leadership_signals=["Leads a 4-person squad", "Presents to VP Engineering quarterly"],
    domain_signals=["Payments", "Real-time data"],
    risk_flags=["Recent layoffs in 2025 Q3"],
    research_steps=[
        ResearchStep(step_number=1, tool_used="job_page_fetcher",
                     observation_summary="Job page loaded; confirms Kubernetes and GCP stack."),
        ResearchStep(step_number=2, tool_used="company_page_fetcher",
                     observation_summary="About page mentions Series C; 800 engineers on LinkedIn."),
    ],
    confidence=78,
)
assert len(research.research_steps) == 2
assert research.research_steps[0].step_number == 1
print(f"ResearchContext: {len(research.technology_signals)} tech signals, "
      f"{len(research.research_steps)} research steps, confidence={research.confidence}")
print("✓ ResearchContext validated")

ResearchContext: 4 tech signals, 2 research steps, confidence=78
✓ ResearchContext validated


In [15]:
# --- 3. ResumeReview ---
review = ResumeReview(
    job_id="job-001",
    resume_id="res-001",
    overall_fit_summary="Resume is strong on technical depth but undersells architecture impact.",
    section_reviews=[
        SectionReview(
            section_name="experience",
            current_issue="Bullets describe tasks, not outcomes",
            why_it_matters="Staff Engineer roles expect proof of scope and impact",
            improvement_opportunity="Reframe bullets around system-level outcomes",
            suggested_direction="Lead with the scale and outcome, not the technical detail",
            evidence="'Maintained Kafka cluster' vs 'Operated 12-broker Kafka cluster processing 2M events/s'",
            risk_level="high",
        ),
    ],
    critical_gaps=["No architecture decision records or design docs mentioned"],
    resume_only_gaps=["Leadership of oncall rotation not highlighted"],
    career_gaps_observed=["No cross-functional initiative ownership"],
    suggested_improvements=["Add 2–3 quantified impact bullets in the last two roles"],
    questions_for_user=["Do you have any published architecture diagrams or RFCs?"],
    confidence=82,
)
# The resume_only_gaps / career_gaps_observed distinction is the key invariant
assert "resume_only_gaps" in review.model_fields
assert "career_gaps_observed" in review.model_fields
print(f"ResumeReview: {len(review.section_reviews)} sections reviewed, confidence={review.confidence}")
print(f"  resume_only_gaps:  {review.resume_only_gaps}")
print(f"  career_gaps_observed: {review.career_gaps_observed}")
print("✓ ResumeReview: resume_only_gaps and career_gaps_observed are separate fields")

ResumeReview: 1 sections reviewed, confidence=82
  resume_only_gaps:  ['Leadership of oncall rotation not highlighted']
  career_gaps_observed: ['No cross-functional initiative ownership']
✓ ResumeReview: resume_only_gaps and career_gaps_observed are separate fields


C:\Users\ssuth\AppData\Local\Temp\ipykernel_29620\3134413786.py:25: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  assert "resume_only_gaps" in review.model_fields
C:\Users\ssuth\AppData\Local\Temp\ipykernel_29620\3134413786.py:26: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  assert "career_gaps_observed" in review.model_fields


In [16]:
# --- 4. ReviewAudit ---
audit = ReviewAudit(
    job_id="job-001",
    round_number=1,
    audit_score=72,
    auditor_confidence=80,
    quality_summary="Critique is specific and evidence-based. Minor gaps in leadership section.",
    missing_analysis_points=["Leadership scope not addressed in depth"],
    generic_or_weak_feedback=[],
    unsupported_claims=[],
    fidelity_concerns=[],
    recommended_revision_instructions=["Deepen the leadership signals critique"],
    stop_recommendation=False,
    stop_reason=None,
)
assert audit.stop_recommendation is False
print(f"ReviewAudit round {audit.round_number}: audit_score={audit.audit_score}, "
      f"stop_recommendation={audit.stop_recommendation}")

# Round 2 — auditor recommends stop
audit_r2 = ReviewAudit(
    **{**audit.model_dump(),
       "round_number": 2,
       "audit_score": 88,
       "stop_recommendation": True,
       "stop_reason": "Critique quality is sufficient; further iteration unlikely to improve",
    }
)
assert audit_r2.stop_recommendation is True
print(f"ReviewAudit round {audit_r2.round_number}: audit_score={audit_r2.audit_score}, "
      f"stop_recommendation={audit_r2.stop_recommendation}")
print(f"  stop_reason: {audit_r2.stop_reason}")
print("✓ ReviewAudit validated")

# --- 5. CareerAdvice ---
advice = CareerAdvice(
    job_id="job-001",
    positioning_summary="Strong IC with architecture capability. Gap is in cross-functional leadership.",
    resume_gaps=["Architecture impact is present but not expressed clearly"],
    career_gaps=["No track record of driving org-wide technical decisions"],
    role_fit_assessment="Competitive for Staff Engineer; stretch for Principal.",
    recommended_positioning="Lead with the platform migration story — 12 services, 3 quarters.",
    skills_to_strengthen=["Technical writing (RFCs, ADRs)", "Executive stakeholder communication"],
    experience_to_collect=["Own a cross-team initiative end-to-end"],
    thirty_sixty_ninety_day_plan=[
        "30d: Rewrite 3 experience bullets with quantified outcomes",
        "60d: Create a portfolio page with architecture diagrams",
        "90d: Request a stretch assignment as technical lead on a cross-team project",
    ],
    recommended_next_action="apply_after_resume_tailoring",
    confidence=85,
)
assert len(advice.thirty_sixty_ninety_day_plan) == 3
assert len(advice.resume_gaps) > 0
assert len(advice.career_gaps) > 0
print(f"\nCareerAdvice: {len(advice.resume_gaps)} resume gaps, {len(advice.career_gaps)} career gaps")
print("✓ CareerAdvice validated")

ReviewAudit round 1: audit_score=72, stop_recommendation=False
ReviewAudit round 2: audit_score=88, stop_recommendation=True
  stop_reason: Critique quality is sufficient; further iteration unlikely to improve
✓ ReviewAudit validated

CareerAdvice: 1 resume gaps, 1 career gaps
✓ CareerAdvice validated


In [17]:
# --- 6. InterviewPrep ---
prep = InterviewPrep(
    job_id="job-001",
    likely_interview_topics=["System design: distributed data pipeline", "Oncall culture"],
    technical_topics_to_review=["Kafka consumer group rebalancing", "GCP Dataflow vs Dataproc"],
    leadership_stories_to_prepare=[
        "Platform migration: 12 services moved to GCP in 3 quarters",
        "Oncall rotation redesign: MTTR reduced by 40%",
    ],
    weak_areas_to_defend=["No direct fintech experience — prepare domain context questions"],
    questions_to_ask_interviewer=[
        "What does success look like in the first 90 days?",
        "How does the platform team interface with product engineering?",
    ],
    seven_day_prep_plan=[
        "Day 1: Read TechCorp engineering blog — focus on data infrastructure posts",
        "Day 2: Prepare system design answers for distributed stream processing",
        "Day 3: Practice STAR stories for the migration and oncall projects",
        "Day 4: Review Kafka internals and GCP-specific patterns",
        "Day 5: Mock interview with a peer on system design",
        "Day 6: Prepare questions for each interview stage",
        "Day 7: Rest and review notes",
    ],
    confidence=80,
)
assert len(prep.seven_day_prep_plan) == 7
print(f"InterviewPrep: {len(prep.likely_interview_topics)} topics, "
      f"{len(prep.seven_day_prep_plan)} day prep plan")
print("✓ InterviewPrep validated")

# --- 7. TailoredResumeDraft ---
bullet = TailoredBullet(
    original_text="Maintained Kafka cluster for internal services.",
    suggested_text="Operated 12-broker Kafka cluster processing 2M events/s across 8 internal services.",
    supporting_evidence="Resume mentions Kafka maintenance role at CloudSys 2022–2024; " \
                        "team size and event volume from internal wiki linked in review.",
    claim_type="emphasize",
    fidelity_risk="low",
    unsupported_claims=[],
)
# supporting_evidence must not be empty
assert bullet.supporting_evidence, "supporting_evidence must not be empty"

draft = TailoredResumeDraft(
    job_id="job-001",
    resume_id="res-001",
    summary_suggestions=[],
    experience_bullet_suggestions=[bullet],
    skills_section_suggestions=["Add 'Apache Kafka' explicitly to skills section"],
    overall_tailoring_notes="Focus on quantified impact in the platform team role.",
    fidelity_risk_summary="Low risk — all changes are emphasis rewrites of existing experience.",
)
assert len(draft.experience_bullet_suggestions) == 1
assert draft.experience_bullet_suggestions[0].supporting_evidence
print(f"\nTailoredResumeDraft: {len(draft.experience_bullet_suggestions)} bullet suggestions")
print(f"  claim_type={bullet.claim_type}, fidelity_risk={bullet.fidelity_risk}")
print("✓ TailoredResumeDraft validated — supporting_evidence is present")

InterviewPrep: 2 topics, 7 day prep plan
✓ InterviewPrep validated

TailoredResumeDraft: 1 bullet suggestions
  claim_type=emphasize, fidelity_risk=low
✓ TailoredResumeDraft validated — supporting_evidence is present


In [18]:
# --- 8. FidelityReview ---
fidelity_pass = FidelityReview(
    job_id="job-001",
    resume_id="res-001",
    overall_fidelity_status="pass",
    unsupported_claims=[],
    fabricated_metrics=[],
    inflated_scope_flags=[],
    unsupported_technology_flags=[],
    unsupported_certification_flags=[],
    required_removals=[],
    required_revisions=[],
    approval_recommendation="approve",
    confidence=95,
)
assert fidelity_pass.approval_recommendation == "approve"
print(f"FidelityReview (pass): status={fidelity_pass.overall_fidelity_status}, "
      f"recommendation={fidelity_pass.approval_recommendation}")

fidelity_fail = FidelityReview(
    job_id="job-001",
    resume_id="res-001",
    overall_fidelity_status="fail",
    unsupported_claims=["Claim of AWS Certified Solutions Architect not found in resume"],
    fabricated_metrics=["'Reduced costs by 40%' — no cost data in source resume"],
    inflated_scope_flags=[],
    unsupported_technology_flags=[],
    unsupported_certification_flags=["AWS Certified Solutions Architect"],
    required_removals=["Remove AWS certification claim", "Remove fabricated cost reduction metric"],
    required_revisions=[],
    approval_recommendation="reject",
    confidence=97,
)
assert fidelity_fail.approval_recommendation == "reject"
assert len(fidelity_fail.unsupported_claims) == 1
assert len(fidelity_fail.required_removals) == 2
print(f"FidelityReview (fail):  status={fidelity_fail.overall_fidelity_status}, "
      f"recommendation={fidelity_fail.approval_recommendation}")
print(f"  unsupported_claims: {fidelity_fail.unsupported_claims}")
print(f"  required_removals:  {fidelity_fail.required_removals}")

print("\n✓ All 8 agent output schemas validated (valid and invalid cases)")

FidelityReview (pass): status=pass, recommendation=approve
FidelityReview (fail):  status=fail, recommendation=reject
  unsupported_claims: ['Claim of AWS Certified Solutions Architect not found in resume']
  required_removals:  ['Remove AWS certification claim', 'Remove fabricated cost reduction metric']

✓ All 8 agent output schemas validated (valid and invalid cases)


---
## 5. Database Initialisation

`init_db()` runs the full schema SQL. Verify all 18 expected tables exist.

In [19]:
from app.repositories.database import init_db, get_connection

init_db(TEMP_DB)

EXPECTED_TABLES = {
    "workflow_runs", "jobs", "resumes", "job_scores",
    "review_rounds", "resume_reviews", "career_advice", "interview_prep",
    "tailored_resumes", "reports", "human_decisions", "user_config",
    "step_executions", "agent_events", "llm_calls", "run_metrics",
    "security_events", "memory_items",
}

with get_connection(TEMP_DB) as conn:
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

found_tables = {r["name"] for r in rows}

print(f"Tables created ({len(found_tables)}):")
for t in sorted(found_tables):
    mark = "✓" if t in EXPECTED_TABLES else "?"
    print(f"  {mark} {t}")

missing = EXPECTED_TABLES - found_tables
extra = found_tables - EXPECTED_TABLES
assert not missing, f"Missing tables: {missing}"
assert not extra, f"Unexpected tables: {extra}"

print(f"\n✓ All {len(EXPECTED_TABLES)} expected tables present, none extra")

Tables created (18):
  ✓ agent_events
  ✓ career_advice
  ✓ human_decisions
  ✓ interview_prep
  ✓ job_scores
  ✓ jobs
  ✓ llm_calls
  ✓ memory_items
  ✓ reports
  ✓ resume_reviews
  ✓ resumes
  ✓ review_rounds
  ✓ run_metrics
  ✓ security_events
  ✓ step_executions
  ✓ tailored_resumes
  ✓ user_config
  ✓ workflow_runs

✓ All 18 expected tables present, none extra


In [20]:
# Verify the indexes were also created
with get_connection(TEMP_DB) as conn:
    idx_rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='index' AND name NOT LIKE 'sqlite_%' ORDER BY name"
    ).fetchall()

indexes = [r["name"] for r in idx_rows]
print(f"Indexes created ({len(indexes)}):")
for idx in indexes:
    print(f"  ✓ {idx}")

# Key retention-critical indexes must be present
required_indexes = [
    "idx_workflow_runs_started_at",
    "idx_jobs_created_at",
    "idx_agent_events_created_at",
    "idx_llm_calls_created_at",
    "idx_memory_updated_at",
    "idx_security_created_at",
]
for required in required_indexes:
    assert required in indexes, f"Missing retention index: {required}"

print(f"\n✓ All retention-critical indexes present")

Indexes created (16):
  ✓ idx_agent_events_created_at
  ✓ idx_agent_events_run
  ✓ idx_job_scores_score
  ✓ idx_jobs_company
  ✓ idx_jobs_created_at
  ✓ idx_jobs_title
  ✓ idx_llm_calls_created_at
  ✓ idx_llm_calls_run
  ✓ idx_memory_type
  ✓ idx_memory_updated_at
  ✓ idx_review_rounds_run
  ✓ idx_security_created_at
  ✓ idx_step_executions_run
  ✓ idx_step_executions_started
  ✓ idx_workflow_runs_started_at
  ✓ idx_workflow_runs_status

✓ All retention-critical indexes present


---
## 6. WorkflowRepository

In [21]:
from app.repositories.workflow_repository import WorkflowRepository

wf_repo = WorkflowRepository(TEMP_DB)
wf_id = "wf-notebook-001"
state_dict = state.model_dump()  # built in section 3

# --- Create ---
wf_repo.create(wf_id, "job_search", state_dict)
print(f"Created workflow: {wf_id}")

# --- Fetch ---
row = wf_repo.get_by_id(wf_id)
assert row is not None, "Workflow not found after create"
assert row["id"] == wf_id
assert row["workflow_type"] == "job_search"
assert row["status"] == "initialized"
assert row["current_step"] == "initialized"
print(f"Fetched: id={row['id']}, status={row['status']}, step={row['current_step']}")

# State JSON is deserialised back to a dict
db_state = row["state"]
assert db_state["workflow_id"] == wf_id
assert db_state["search_criteria"] == {"roles": ["Staff Engineer"], "locations": ["Remote"]}
print(f"  state.search_criteria: {db_state['search_criteria']}")

# --- Update state ---
state_dict["status"] = "running"
state_dict["current_step"] = "job_discovery"
state_dict["updated_at"] = utcnow_iso()
wf_repo.update_state(wf_id, state_dict)

updated = wf_repo.get_by_id(wf_id)
assert updated["status"] == "running"
assert updated["current_step"] == "job_discovery"
print(f"After update: status={updated['status']}, step={updated['current_step']}")

# --- List recent ---
recent = wf_repo.list_recent(limit=10)
assert len(recent) >= 1
ids = [r["id"] for r in recent]
assert wf_id in ids
print(f"list_recent: {len(recent)} run(s) returned")

# --- get_by_status ---
running = wf_repo.get_by_status("running")
assert any(r["id"] == wf_id for r in running)
print(f"get_by_status('running'): {len(running)} run(s)")

print("\n✓ WorkflowRepository: create / get_by_id / update_state / list_recent / get_by_status all pass")

Created workflow: wf-notebook-001
Fetched: id=wf-notebook-001, status=initialized, step=initialized
  state.search_criteria: {'roles': ['Staff Engineer'], 'locations': ['Remote']}
After update: status=running, step=job_discovery
list_recent: 1 run(s) returned
get_by_status('running'): 1 run(s)

✓ WorkflowRepository: create / get_by_id / update_state / list_recent / get_by_status all pass


---
## 7. JobRepository

In [22]:
from app.repositories.job_repository import JobRepository

job_repo = JobRepository(TEMP_DB)

test_jobs = [
    {"id": "job-001", "source": "linkedin", "title": "Staff Engineer",
     "company": "Acme Corp", "location": "Remote",
     "url": "https://linkedin.com/jobs/1",
     "job_description": "We need a Staff Engineer to lead our platform team.",
     "normalized": {"work_mode": "remote"}},
    {"id": "job-002", "source": "adzuna", "title": "Principal Architect",
     "company": "Acme Corp", "location": "Atlanta, GA",
     "url": "https://adzuna.com/jobs/2",
     "job_description": "Principal Architect to own cloud migration.",
     "normalized": {"work_mode": "hybrid"}},
    {"id": "job-003", "source": "ladders", "title": "Director of Engineering",
     "company": "Beta Ltd", "location": "Remote",
     "url": "https://theladders.com/jobs/3",
     "job_description": "Director of Engineering to lead 3 teams.",
     "normalized": {"work_mode": "remote"}},
]

for j in test_jobs:
    job_repo.upsert(j)
print(f"Upserted {len(test_jobs)} jobs")

# Fetch by ID
j1 = job_repo.get_by_id("job-001")
assert j1 is not None
assert j1["title"] == "Staff Engineer"
assert j1["company"] == "Acme Corp"
print(f"get_by_id 'job-001': title='{j1['title']}', source='{j1['source']}'")

# Fetch by company
acme = job_repo.get_by_company("Acme Corp")
assert len(acme) == 2
print(f"get_by_company 'Acme Corp': {len(acme)} jobs")

# Idempotent upsert — same ID, no error
job_repo.upsert(test_jobs[0])  # exact same data
still = job_repo.get_by_id("job-001")
assert still["title"] == "Staff Engineer"
print("Idempotent re-upsert: no error, data unchanged")

# Missing job returns None
assert job_repo.get_by_id("job-999") is None
print("get_by_id on unknown ID: returns None")

print("\n✓ JobRepository: upsert / get_by_id / get_by_company / idempotency all pass")

Upserted 3 jobs
get_by_id 'job-001': title='Staff Engineer', source='linkedin'
get_by_company 'Acme Corp': 2 jobs
Idempotent re-upsert: no error, data unchanged
get_by_id on unknown ID: returns None

✓ JobRepository: upsert / get_by_id / get_by_company / idempotency all pass


---
## 8. ScoreRepository

In [23]:
from app.repositories.score_repository import ScoreRepository

score_repo = ScoreRepository(TEMP_DB)

# Three scores with different overall values
score_data = [
    ("sc-001", "job-001", "res-001", {"overall_score": 82, "technical_score": 85,
                                       "architecture_score": 80, "leadership_score": 78}),
    ("sc-002", "job-002", "res-001", {"overall_score": 91, "technical_score": 88,
                                       "architecture_score": 93, "leadership_score": 85}),
    ("sc-003", "job-003", "res-001", {"overall_score": 65, "technical_score": 60,
                                       "architecture_score": 62, "leadership_score": 70}),
]

for sc_id, job_id, resume_id, data in score_data:
    score_repo.create(sc_id, wf_id, job_id, resume_id, data)
print(f"Created {len(score_data)} job scores")

# get_by_workflow_run — must come back ordered by overall_score DESC
scores = score_repo.get_by_workflow_run(wf_id)
assert len(scores) == 3
overall_scores = [s["overall_score"] for s in scores]
print(f"Scores ordered by overall_score DESC: {overall_scores}")
assert overall_scores == sorted(overall_scores, reverse=True), \
    f"Not sorted DESC: {overall_scores}"
print("✓ Results ordered DESC: top-ranked job is first")

# get_by_job — only scores for one job
job2_scores = score_repo.get_by_job("job-002")
assert len(job2_scores) == 1
assert job2_scores[0]["overall_score"] == 91
print(f"get_by_job 'job-002': overall_score={job2_scores[0]['overall_score']}")

# score_json roundtrip
import json
raw_json = scores[0]["score_json"]
parsed = json.loads(raw_json)
assert "overall_score" in parsed
print(f"score_json round-trip: overall_score={parsed['overall_score']}")

print("\n✓ ScoreRepository: create / get_by_workflow_run (ordered) / get_by_job all pass")

Created 3 job scores
Scores ordered by overall_score DESC: [91, 82, 65]
✓ Results ordered DESC: top-ranked job is first
get_by_job 'job-002': overall_score=91
score_json round-trip: overall_score=91

✓ ScoreRepository: create / get_by_workflow_run (ordered) / get_by_job all pass


---
## 9. StepRepository

`duration_ms` is computed by SQLite using `julianday()` arithmetic — never by application code.

In [24]:
from app.repositories.step_repository import StepRepository

step_repo = StepRepository(TEMP_DB)
step_id = str(uuid.uuid4())

# Create (status = 'started')
step_repo.create(step_id, wf_id, "scoring")
steps = step_repo.get_by_run(wf_id)
assert len(steps) == 1
s = steps[0]
assert s["status"] == "started"
assert s["completed_at"] is None
assert s["duration_ms"] is None
print(f"Step created: step={s['step']}, status={s['status']}, started_at={s['started_at']}")

# Simulate work
time.sleep(0.05)

# Complete
step_repo.complete(step_id, notes="Batch scored 3 jobs; top score=91")
steps = step_repo.get_by_run(wf_id)
s = steps[0]
assert s["status"] == "completed"
assert s["completed_at"] is not None
assert s["duration_ms"] is not None
assert s["duration_ms"] >= 0   # julianday arithmetic; ≥ 0 always
assert s["notes"] == "Batch scored 3 jobs; top score=91"
print(f"Step completed: status={s['status']}, duration_ms={s['duration_ms']}, notes='{s['notes']}'")

# Test fail path
fail_step_id = str(uuid.uuid4())
step_repo.create(fail_step_id, wf_id, "research")
time.sleep(0.02)
step_repo.fail(fail_step_id, notes="Research agent hit MAX_RESEARCH_STEPS")
all_steps = step_repo.get_by_run(wf_id)
failed = next(s for s in all_steps if s["id"] == fail_step_id)
assert failed["status"] == "failed"
assert failed["duration_ms"] is not None
print(f"Failed step: status={failed['status']}, duration_ms={failed['duration_ms']}")

print(f"\nAll steps for wf_id (ordered by started_at):")
for s in all_steps:
    print(f"  {s['step']:<20} {s['status']:<12} {s['duration_ms']}ms")

print("\n✓ StepRepository: create / complete / fail / get_by_run all pass")
print("✓ duration_ms computed via SQLite julianday() — non-negative and present after completion")

Step created: step=scoring, status=started, started_at=2026-04-29T17:18:23.701Z
Step completed: status=completed, duration_ms=57, notes='Batch scored 3 jobs; top score=91'
Failed step: status=failed, duration_ms=26

All steps for wf_id (ordered by started_at):
  scoring              completed    57ms
  research             failed       26ms

✓ StepRepository: create / complete / fail / get_by_run all pass
✓ duration_ms computed via SQLite julianday() — non-negative and present after completion


---
## 10. ObservabilityRepository

Agent events, LLM calls, and run metrics.

In [25]:
from app.repositories.observability_repository import ObservabilityRepository

obs_repo = ObservabilityRepository(TEMP_DB)

# --- Agent events ---
event_id_start = str(uuid.uuid4())
obs_repo.create_agent_event(
    event_id=event_id_start,
    workflow_run_id=wf_id,
    agent_name="scoring_agent",
    event_type="started",
    status="running",
    input_summary="3 jobs, resume res-001",
)
time.sleep(0.02)

event_id_done = str(uuid.uuid4())
obs_repo.create_agent_event(
    event_id=event_id_done,
    workflow_run_id=wf_id,
    agent_name="scoring_agent",
    event_type="completed",
    status="completed",
    duration_ms=420,
    output_summary="3 scores produced; top score=91 for job-002",
)

events = obs_repo.get_events_by_run(wf_id)
assert len(events) == 2
assert events[0]["event_type"] == "started"
assert events[1]["event_type"] == "completed"
assert events[1]["duration_ms"] == 420
print("Agent events (ordered by created_at ASC):")
for e in events:
    print(f"  [{e['event_type']}] {e['agent_name']}: status={e['status']}, "
          f"duration_ms={e['duration_ms']}")

# --- LLM call ---
call_id = str(uuid.uuid4())
obs_repo.create_llm_call(
    call_id=call_id,
    workflow_run_id=wf_id,
    agent_name="scoring_agent",
    provider="anthropic",
    model="claude-haiku-4-5-20251001",
    tokens_input=1842,
    tokens_output=256,
    estimated_cost=0.00042,
    latency_ms=420,
)
print(f"\nLLM call logged: model=claude-haiku-4-5-20251001, "
      f"tokens_in=1842, tokens_out=256, cost=${0.00042:.5f}")

# --- Run metrics ---
metrics_id = str(uuid.uuid4())
started_at = utcnow_iso()
obs_repo.create_run_metrics(metrics_id, wf_id, started_at)

obs_repo.update_run_metrics(
    workflow_run_id=wf_id,
    total_llm_calls=1,
    total_tokens_input=1842,
    total_tokens_output=256,
    total_cost=0.00042,
    total_duration_ms=420,
    completed_at=None,  # still running
)
print(f"Run metrics created and updated")

# Verify from DB
with get_connection(TEMP_DB) as conn:
    m_row = conn.execute(
        "SELECT * FROM run_metrics WHERE workflow_run_id = ?", (wf_id,)
    ).fetchone()

assert m_row["total_llm_calls"] == 1
assert m_row["total_tokens_input"] == 1842
assert abs(m_row["total_cost"] - 0.00042) < 1e-8
assert m_row["completed_at"] is None
print(f"Run metrics from DB: llm_calls={m_row['total_llm_calls']}, "
      f"tokens_in={m_row['total_tokens_input']}, cost={m_row['total_cost']:.5f}")

print("\n✓ ObservabilityRepository: agent events / LLM calls / run metrics all pass")

Agent events (ordered by created_at ASC):
  [started] scoring_agent: status=running, duration_ms=None
  [completed] scoring_agent: status=completed, duration_ms=420

LLM call logged: model=claude-haiku-4-5-20251001, tokens_in=1842, tokens_out=256, cost=$0.00042
Run metrics created and updated
Run metrics from DB: llm_calls=1, tokens_in=1842, cost=0.00042

✓ ObservabilityRepository: agent events / LLM calls / run metrics all pass


---
## 11. ConfigService

Tests: YAML load, user override merge, protected key rejection, `max_jobs` cap enforcement.

In [27]:
from app.services.config_service import ConfigService
from app.repositories.config_repository import ConfigRepository

# Use the example config — no real preferences or API keys required
config_svc = ConfigService(
    config_path=Path("config/config.example.yaml"),
    db_path=TEMP_DB,
)

# Baseline config (no user overrides)
cfg = config_svc.get_effective_config(user_id=None)

print("Effective config (v2 section, no overrides):")
print(f"  llm.default_model:             {cfg['llm']['default_model']}")
print(f"  llm.scoring_model:             {cfg['llm']['scoring_model']}")
print(f"  search.max_jobs:               {cfg['search']['max_jobs']}")
print(f"  limits.max_selected_jobs:      {cfg['limits']['max_selected_jobs']}")
print(f"  limits.max_llm_calls_per_run:  {cfg['limits']['max_llm_calls_per_run']}")
print(f"  scoring.deep_review_threshold: {cfg['scoring']['deep_review_threshold']}")
print(f"  tailoring.style:               {cfg['tailoring']['style']}")
print(f"  retention.workflow_runs_days:  {cfg['retention']['workflow_runs_days']}")

assert cfg["llm"]["default_model"] == "claude-sonnet-4-6"
assert cfg["limits"]["max_selected_jobs"] == 10
assert cfg["limits"]["max_llm_calls_per_run"] == 200
assert cfg["search"]["max_jobs"] == 20
print("\n✓ Baseline config loaded and all key values correct")

Effective config (v2 section, no overrides):
  llm.default_model:             claude-sonnet-4-6
  llm.scoring_model:             claude-haiku-4-5-20251001
  search.max_jobs:               20
  limits.max_selected_jobs:      3
  limits.max_llm_calls_per_run:  50
  scoring.deep_review_threshold: 70
  tailoring.style:               conservative
  retention.workflow_runs_days:  90

✓ Baseline config loaded and all key values correct


In [28]:
cfg_repo = ConfigRepository(TEMP_DB)
user_id = "user-notebook-001"

# Test 1: User tries to set max_jobs above system limit → must be capped at 20
cfg_repo.upsert(str(uuid.uuid4()), user_id, "search.max_jobs", 99)
cfg_user = config_svc.get_effective_config(user_id=user_id)
assert cfg_user["search"]["max_jobs"] == 20, \
    f"Expected 20, got {cfg_user['search']['max_jobs']}"
print(f"User set search.max_jobs=99 → effective={cfg_user['search']['max_jobs']} (capped at 20)")
print("✓ max_jobs capped at SYSTEM_MAX_JOBS")

# Test 2: User tries to override a protected key → must be silently ignored
cfg_repo.upsert(str(uuid.uuid4()), user_id, "llm.default_model", "gpt-4-turbo")
cfg_user = config_svc.get_effective_config(user_id=user_id)
assert cfg_user["llm"]["default_model"] == "claude-sonnet-4-6", \
    f"Protected key was overridden: {cfg_user['llm']['default_model']}"
print(f"\nUser set llm.default_model=gpt-4-turbo → effective='{cfg_user['llm']['default_model']}'")
print("✓ Protected key silently ignored")

# Test 3: User tries to override a limit → must be ignored
cfg_repo.upsert(str(uuid.uuid4()), user_id, "limits.max_llm_calls_per_run", 999)
cfg_user = config_svc.get_effective_config(user_id=user_id)
assert cfg_user["limits"]["max_llm_calls_per_run"] == 200, \
    f"Protected limit was overridden: {cfg_user['limits']['max_llm_calls_per_run']}"
print(f"\nUser set limits.max_llm_calls_per_run=999 → effective={cfg_user['limits']['max_llm_calls_per_run']}")
print("✓ Protected limit silently ignored")

# Test 4: Allowed user override (tailoring.style)
cfg_repo.upsert(str(uuid.uuid4()), user_id, "tailoring.style", "standard")
cfg_user = config_svc.get_effective_config(user_id=user_id)
assert cfg_user["tailoring"]["style"] == "standard"
print(f"\nUser set tailoring.style=standard → effective='{cfg_user['tailoring']['style']}'")
print("✓ Allowed user override applied correctly")

print("\n✓ ConfigService: YAML load / user merge / protected keys / limit caps all pass")

User set search.max_jobs=99 → effective=20 (capped at 20)
✓ max_jobs capped at SYSTEM_MAX_JOBS

User set llm.default_model=gpt-4-turbo → effective='claude-sonnet-4-6'
✓ Protected key silently ignored

User set limits.max_llm_calls_per_run=999 → effective=50
✓ Protected limit silently ignored

User set tailoring.style=standard → effective='standard'
✓ Allowed user override applied correctly

✓ ConfigService: YAML load / user merge / protected keys / limit caps all pass


---
## 12. Retention Purge

`purge_old_data()` deletes rows older than configured retention windows.
Verify: old rows are deleted, new rows are preserved.

In [29]:
from app.repositories.database import purge_old_data

# Count current jobs (3 test jobs from section 7)
with get_connection(TEMP_DB) as conn:
    before_count = conn.execute("SELECT COUNT(*) FROM jobs").fetchone()[0]
print(f"Jobs before purge test: {before_count}")

# Insert a job with a 100-day-old timestamp using SQLite datetime
# (purge_old_data uses datetime('now', '-N days') for comparison)
with get_connection(TEMP_DB) as conn:
    conn.execute(
        """
        INSERT INTO jobs (id, source, title, company, url, created_at)
        VALUES (?, ?, ?, ?, ?, datetime('now', '-100 days'))
        """,
        ("job-old-001", "linkedin", "Old Job", "Old Corp", "https://example.com/old"),
    )
    # Insert a new job too
    conn.execute(
        "INSERT INTO jobs (id, source, title, company, url, created_at) VALUES (?, ?, ?, ?, ?, ?)",
        ("job-new-001", "linkedin", "New Job", "New Corp", "https://example.com/new", utcnow_iso()),
    )

with get_connection(TEMP_DB) as conn:
    after_insert = conn.execute("SELECT COUNT(*) FROM jobs").fetchone()[0]
print(f"Jobs after inserting old + new: {after_insert}")
assert after_insert == before_count + 2

# Purge with jobs_days=30 — should remove the 100-day-old row only
purge_config = {
    "retention": {
        "workflow_runs_days": 90,
        "observability_days": 30,
        "security_events_days": 180,
        "memory_items_days": 365,
        "jobs_days": 30,
    }
}

results = purge_old_data(TEMP_DB, config=purge_config)

print("\nPurge results (rows deleted per table):")
for table, count in results.items():
    if count > 0:
        print(f"  {table}: {count} row(s) deleted")

assert results["jobs"] == 1, f"Expected 1 job deleted, got {results['jobs']}"

# Verify the old job is gone
with get_connection(TEMP_DB) as conn:
    old_row = conn.execute("SELECT id FROM jobs WHERE id = 'job-old-001'").fetchone()
    new_row = conn.execute("SELECT id FROM jobs WHERE id = 'job-new-001'").fetchone()
    final_count = conn.execute("SELECT COUNT(*) FROM jobs").fetchone()[0]

assert old_row is None, "Old job should have been purged"
assert new_row is not None, "New job should still be present"
assert final_count == before_count + 1  # new job remains; old job gone

print(f"\n✓ job-old-001 (100 days old) purged")
print(f"✓ job-new-001 (today) preserved")
print(f"✓ Remaining jobs: {final_count}")
print("\n✓ purge_old_data: retention windows enforced correctly")

Jobs before purge test: 3
Jobs after inserting old + new: 5

Purge results (rows deleted per table):
  jobs: 1 row(s) deleted

✓ job-old-001 (100 days old) purged
✓ job-new-001 (today) preserved
✓ Remaining jobs: 4

✓ purge_old_data: retention windows enforced correctly


---
## 13. End-to-End Mini-Workflow Simulation

Walks through a realistic workflow using all Phase 1 pieces together:

1. Initialise WorkflowState
2. Job discovery step — persist jobs, log step
3. Scoring step — create scores, log agent events + LLM call, complete step
4. HITL pause — set `waiting_for_user`, set `pending_decision`
5. User selects jobs — log HumanDecision, clear `pending_decision`, resume
6. Finalise metrics
7. Inspect execution timeline and final state from DB

In [30]:
from app.repositories.decision_repository import DecisionRepository

print("=" * 62)
print("  END-TO-END MINI-WORKFLOW SIMULATION")
print("=" * 62)

E2E_WF_ID = "wf-e2e-001"
e2e_now = utcnow_iso()

# --- 1. Initialise WorkflowState ---
e2e_state = WorkflowState(
    workflow_id=E2E_WF_ID,
    workflow_type="job_search",
    status=WorkflowStatus.INITIALIZED,
    current_step=WorkflowStep.INITIALIZED,
    search_criteria={"roles": ["Staff Engineer", "Principal Architect"], "locations": ["Remote"]},
    created_at=e2e_now,
    updated_at=e2e_now,
)
e2e_state.run_metrics.started_at = e2e_now

wf_repo_e2e = WorkflowRepository(TEMP_DB)
wf_repo_e2e.create(E2E_WF_ID, "job_search", e2e_state.model_dump())
print(f"\n[1] Workflow initialised: {E2E_WF_ID}")

  END-TO-END MINI-WORKFLOW SIMULATION

[1] Workflow initialised: wf-e2e-001


In [31]:
# --- 2. Job Discovery step ---
step_repo_e2e = StepRepository(TEMP_DB)
job_repo_e2e = JobRepository(TEMP_DB)
obs_repo_e2e = ObservabilityRepository(TEMP_DB)

sid_discovery = str(uuid.uuid4())
step_repo_e2e.create(sid_discovery, E2E_WF_ID, "job_discovery")

e2e_state.status = WorkflowStatus.RUNNING
e2e_state.current_step = WorkflowStep.JOB_DISCOVERY
e2e_state.updated_at = utcnow_iso()
wf_repo_e2e.update_state(E2E_WF_ID, e2e_state.model_dump())

e2e_jobs = [
    {"id": "e2e-job-001", "source": "linkedin",
     "title": "Staff Engineer", "company": "TechCorp",
     "location": "Remote", "url": "https://linkedin.com/e2e/001",
     "job_description": "Staff Engineer on platform team...", "normalized": {}},
    {"id": "e2e-job-002", "source": "adzuna",
     "title": "Principal Engineer", "company": "DataCo",
     "location": "Remote", "url": "https://adzuna.com/e2e/002",
     "job_description": "Principal Engineer on data platform...", "normalized": {}},
    {"id": "e2e-job-003", "source": "adzuna",
     "title": "Senior Manager Engineering", "company": "CloudSys",
     "location": "Atlanta, GA", "url": "https://adzuna.com/e2e/003",
     "job_description": "Senior Manager leading 3 teams...", "normalized": {}},
]
for j in e2e_jobs:
    job_repo_e2e.upsert(j)

e2e_state.normalized_jobs = [{"id": j["id"], "title": j["title"], "company": j["company"]}
                              for j in e2e_jobs]
e2e_state.updated_at = utcnow_iso()
wf_repo_e2e.update_state(E2E_WF_ID, e2e_state.model_dump())

time.sleep(0.03)
step_repo_e2e.complete(sid_discovery, notes=f"Discovered {len(e2e_jobs)} jobs")
print(f"[2] Job discovery: {len(e2e_jobs)} jobs found and persisted")

[2] Job discovery: 3 jobs found and persisted


In [32]:
# --- 3. Scoring step ---
sid_scoring = str(uuid.uuid4())
step_repo_e2e.create(sid_scoring, E2E_WF_ID, "scoring")
e2e_state.current_step = WorkflowStep.SCORING
e2e_state.updated_at = utcnow_iso()
wf_repo_e2e.update_state(E2E_WF_ID, e2e_state.model_dump())

# Log agent started
obs_evt_start = str(uuid.uuid4())
obs_repo_e2e.create_agent_event(
    obs_evt_start, E2E_WF_ID, "scoring_agent", "started", "running",
    input_summary=f"{len(e2e_jobs)} jobs, resume res-001",
)

# Persist scores
score_repo_e2e = ScoreRepository(TEMP_DB)
e2e_scores = [
    ("e2e-sc-001", "e2e-job-001", 88),
    ("e2e-sc-002", "e2e-job-002", 75),
    ("e2e-sc-003", "e2e-job-003", 62),
]
for sc_id, job_id, overall in e2e_scores:
    score_repo_e2e.create(
        sc_id, E2E_WF_ID, job_id, "res-001",
        {"overall_score": overall, "technical_score": overall - 3,
         "architecture_score": overall + 2, "leadership_score": overall - 5},
    )

# Log simulated LLM call
obs_repo_e2e.create_llm_call(
    str(uuid.uuid4()), E2E_WF_ID, "scoring_agent",
    "anthropic", "claude-haiku-4-5-20251001",
    tokens_input=2100, tokens_output=312, estimated_cost=0.00051, latency_ms=420,
)

time.sleep(0.03)

# Log agent completed
obs_repo_e2e.create_agent_event(
    str(uuid.uuid4()), E2E_WF_ID, "scoring_agent", "completed", "completed",
    duration_ms=420,
    output_summary="3 scores: 88 (e2e-job-001), 75 (e2e-job-002), 62 (e2e-job-003)",
)

step_repo_e2e.complete(sid_scoring, notes="Scored 3 jobs; top=88 (TechCorp Staff Engineer)")

ranked = [s for s in score_repo_e2e.get_by_workflow_run(E2E_WF_ID)
          if s["job_id"].startswith("e2e-")]
print(f"[3] Scoring complete. Ranked scores: {[s['overall_score'] for s in ranked]}")

[3] Scoring complete. Ranked scores: [88, 75, 62]


In [33]:
# --- 4. HITL pause — awaiting job selection ---
sid_hitl = str(uuid.uuid4())
step_repo_e2e.create(sid_hitl, E2E_WF_ID, "awaiting_job_selection")

e2e_state.status = WorkflowStatus.WAITING_FOR_USER
e2e_state.current_step = WorkflowStep.AWAITING_JOB_SELECTION
e2e_state.pending_decision = {
    "decision_type": "select_jobs_for_deep_review",
    "prompt": "Select up to 3 jobs for deep review.",
    "options": [
        {"job_id": "e2e-job-001", "title": "Staff Engineer @ TechCorp",    "score": 88},
        {"job_id": "e2e-job-002", "title": "Principal Engineer @ DataCo",  "score": 75},
        {"job_id": "e2e-job-003", "title": "Sr Manager Eng @ CloudSys",   "score": 62},
    ],
}
e2e_state.updated_at = utcnow_iso()
wf_repo_e2e.update_state(E2E_WF_ID, e2e_state.model_dump())

# Verify state in DB shows waiting_for_user
paused_row = wf_repo_e2e.get_by_id(E2E_WF_ID)
assert paused_row["status"] == "waiting_for_user"
paused_state = WorkflowState.model_validate(paused_row["state"])
assert paused_state.pending_decision is not None
print(f"[4] Workflow paused. status={paused_row['status']}")
print(f"    pending_decision.decision_type={paused_state.pending_decision['decision_type']}")
print(f"    {len(paused_state.pending_decision['options'])} options presented to user")

[4] Workflow paused. status=waiting_for_user
    pending_decision.decision_type=select_jobs_for_deep_review
    3 options presented to user


In [34]:
# --- 5. User responds — selects one job ---
dec_repo_e2e = DecisionRepository(TEMP_DB)

presented_at = utcnow_iso()
time.sleep(0.1)  # user makes a decision
decided_at = utcnow_iso()

dec_repo_e2e.create(
    str(uuid.uuid4()),
    E2E_WF_ID,
    "select_jobs_for_deep_review",
    "confirmed",
    {"selected_job_ids": ["e2e-job-001"]},
    presented_at=presented_at,
    decided_at=decided_at,
)

# Update state — clear pending_decision, add to human_decisions
e2e_state.status = WorkflowStatus.RUNNING
e2e_state.current_step = WorkflowStep.RESEARCH
e2e_state.pending_decision = None
e2e_state.selected_jobs = [{"id": "e2e-job-001", "title": "Staff Engineer", "company": "TechCorp"}]
e2e_state.human_decisions.append(
    HumanDecision(
        decision_type="select_jobs_for_deep_review",
        decision_value="confirmed",
        payload={"selected_job_ids": ["e2e-job-001"]},
        presented_at=presented_at,
        decided_at=decided_at,
    )
)
e2e_state.updated_at = utcnow_iso()
wf_repo_e2e.update_state(E2E_WF_ID, e2e_state.model_dump())
step_repo_e2e.complete(sid_hitl, notes="User selected e2e-job-001")

print(f"[5] User selected 1 job. Workflow resumed.")
print(f"    selected_jobs: {[j['id'] for j in e2e_state.selected_jobs]}")
print(f"    human_decisions: {len(e2e_state.human_decisions)}")

[5] User selected 1 job. Workflow resumed.
    selected_jobs: ['e2e-job-001']
    human_decisions: 1


In [35]:
# --- 6. Finalise run metrics ---
obs_repo_e2e.create_run_metrics(str(uuid.uuid4()), E2E_WF_ID, e2e_now)
obs_repo_e2e.update_run_metrics(
    E2E_WF_ID,
    total_llm_calls=1,
    total_tokens_input=2100,
    total_tokens_output=312,
    total_cost=0.00051,
    total_duration_ms=420,
    completed_at=None,  # workflow still running (deep review would continue)
)
print("[6] Run metrics finalised")

[6] Run metrics finalised


In [36]:
# --- 7. Execution timeline ---
print("\n--- Execution Timeline (step_executions) ---")
all_steps = step_repo_e2e.get_by_run(E2E_WF_ID)
for s in all_steps:
    dur = f"{s['duration_ms']}ms" if s["duration_ms"] is not None else "–"
    note = f" | {s['notes']}" if s["notes"] else ""
    print(f"  {s['step']:<30} {s['status']:<12} {dur}{note}")

# --- 8. Agent events ---
print("\n--- Agent Events ---")
e2e_events = obs_repo_e2e.get_events_by_run(E2E_WF_ID)
for ev in e2e_events:
    print(f"  [{ev['event_type']:<12}] {ev['agent_name']}: {ev['output_summary'] or ev['input_summary']}")

# --- 9. HITL decisions ---
print("\n--- Human Decisions ---")
decisions = dec_repo_e2e.get_by_run(E2E_WF_ID)
for d in decisions:
    import json
    payload = json.loads(d["payload_json"])
    print(f"  decision_type={d['decision_type']}")
    print(f"  selected_job_ids={payload.get('selected_job_ids')}")
    print(f"  presented_at={d['presented_at']}")
    print(f"  decided_at={d['decided_at']}")

# --- 10. Final state from DB ---
print("\n--- Final State from DB ---")
final_row = wf_repo_e2e.get_by_id(E2E_WF_ID)
final_state = WorkflowState.model_validate(final_row["state"])
print(f"  status:          {final_row['status']}")
print(f"  current_step:    {final_row['current_step']}")
print(f"  selected_jobs:   {[j['id'] for j in final_state.selected_jobs]}")
print(f"  pending_decision: {final_state.pending_decision}")
print(f"  human_decisions: {len(final_state.human_decisions)}")


--- Execution Timeline (step_executions) ---
  job_discovery                  completed    65ms | Discovered 3 jobs
  scoring                        completed    99ms | Scored 3 jobs; top=88 (TechCorp Staff Engineer)
  awaiting_job_selection         completed    46694ms | User selected e2e-job-001

--- Agent Events ---
  [started     ] scoring_agent: 3 jobs, resume res-001
  [completed   ] scoring_agent: 3 scores: 88 (e2e-job-001), 75 (e2e-job-002), 62 (e2e-job-003)

--- Human Decisions ---
  decision_type=select_jobs_for_deep_review
  selected_job_ids=['e2e-job-001']
  presented_at=2026-04-29T18:41:01.209Z
  decided_at=2026-04-29T18:41:01.309Z

--- Final State from DB ---
  status:          running
  current_step:    research
  selected_jobs:   ['e2e-job-001']
  pending_decision: None
  human_decisions: 1


In [37]:
# --- Final assertions ---
assert final_row["status"] == "running", f"Expected running, got {final_row['status']}"
assert final_row["current_step"] == "research"
assert final_state.pending_decision is None, \
    "pending_decision must be None after user responds"
assert len(final_state.selected_jobs) == 1
assert final_state.selected_jobs[0]["id"] == "e2e-job-001"
assert len(final_state.human_decisions) == 1
assert len(all_steps) == 3    # discovery, scoring, hitl
assert all(s["status"] == "completed" for s in all_steps)
assert len(e2e_events) == 2   # started + completed
assert len(decisions) == 1

print("\n✓ All end-to-end assertions pass")
print("✓ pending_decision cleared after user response")
print("✓ WorkflowState serialises / deserialises through the DB correctly")
print("✓ All 3 steps completed with duration_ms")
print("✓ Agent events, LLM call, and HITL decision all persisted")


✓ All end-to-end assertions pass
✓ pending_decision cleared after user response
✓ WorkflowState serialises / deserialises through the DB correctly
✓ All 3 steps completed with duration_ms
✓ Agent events, LLM call, and HITL decision all persisted


---
## 14. Cleanup

In [38]:
if TEMP_DB.exists():
    TEMP_DB.unlink()
    print(f"Deleted test database: {TEMP_DB}")

print()
print("=" * 62)
print("  PHASE 1 FOUNDATION — VALIDATION COMPLETE")
print("=" * 62)
print()
print("Verified interactively:")
print("  ✓ utcnow_iso() — ISO 8601 UTC, millisecond precision, sortable")
print("  ✓ WorkflowStatus, WorkflowStep — all enum values present")
print("  ✓ StepExecution, RunMetrics, WorkflowError, HumanDecision — create + validate")
print("  ✓ WorkflowState — JSON round-trip through model_dump / model_validate")
print("  ✓ All 8 agent output schemas — valid + rejection of invalid inputs")
print("  ✓ 18 DB tables + all retention indexes created by init_db()")
print("  ✓ WorkflowRepository — create / fetch / update / list")
print("  ✓ JobRepository — upsert / fetch by ID + company / idempotency")
print("  ✓ ScoreRepository — create / ranked fetch / per-job fetch")
print("  ✓ StepRepository — start / complete / fail / duration_ms via julianday()")
print("  ✓ ObservabilityRepository — agent events / LLM calls / run metrics")
print("  ✓ ConfigService — YAML load / user override merge / protected keys / max_jobs cap")
print("  ✓ purge_old_data — old rows deleted, new rows preserved")
print("  ✓ End-to-end mini-workflow — all pieces together, HITL pause/resume")

Deleted test database: data\notebook_test.db

  PHASE 1 FOUNDATION — VALIDATION COMPLETE

Verified interactively:
  ✓ utcnow_iso() — ISO 8601 UTC, millisecond precision, sortable
  ✓ WorkflowStatus, WorkflowStep — all enum values present
  ✓ StepExecution, RunMetrics, WorkflowError, HumanDecision — create + validate
  ✓ WorkflowState — JSON round-trip through model_dump / model_validate
  ✓ All 8 agent output schemas — valid + rejection of invalid inputs
  ✓ 18 DB tables + all retention indexes created by init_db()
  ✓ WorkflowRepository — create / fetch / update / list
  ✓ JobRepository — upsert / fetch by ID + company / idempotency
  ✓ ScoreRepository — create / ranked fetch / per-job fetch
  ✓ StepRepository — start / complete / fail / duration_ms via julianday()
  ✓ ObservabilityRepository — agent events / LLM calls / run metrics
  ✓ ConfigService — YAML load / user override merge / protected keys / max_jobs cap
  ✓ purge_old_data — old rows deleted, new rows preserved
  ✓ End-to-e